In [ ]:
%pip install requests beautifulsoup4 pandas

: 

In [ ]:
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

URL = "https://www.niddk.nih.gov/health-information/professionals/clinical-tools-patient-management/kidney-disease/identify-manage-patients/manage-ckd/slow-progression-reduce-complications"
headers = {"User-Agent": "AcademicNutritionResearch/1.0 (educational, respectful requests)"}
response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")
print('HTTP:', response.status_code, '| page:', soup.title.get_text(' ', strip=True) if soup.title else '(untitled)')

: 

In [ ]:
heading = soup.find(lambda tag: tag.name in ('h2','h3','h4') and
                    'nutritional interventions' in tag.get_text(' ', strip=True).lower())
if heading is None:
    raise RuntimeError('Nutritional Interventions heading not found. Inspect the site HTML; do not guess values.')
chunks = []
for element in heading.next_elements:
    if getattr(element, 'name', None) in ('h2','h3','h4') and element is not heading:
        break
    if getattr(element, 'name', None) in ('p','li'):
        text = element.get_text(' ', strip=True)
        if text and text not in chunks:
            chunks.append(text)
section_text = '\n'.join(chunks)
print(section_text[:5000])
assert 'sodium' in section_text.lower() and 'protein' in section_text.lower(), 'Unexpected page structure' 

In [ ]:
sodium_match = re.search(r'Limit dietary sodium to\s*([\d,]+)\s*mg/day', section_text, re.I)
protein_match = re.search(r'Adequate protein intake is\s*([\d.]+)\s*g/kg(?:\s*body weight)?/day', section_text, re.I)

if not sodium_match:
    raise RuntimeError('Sodium limit text changed. Manually review the source before exporting.')

sodium_limit = float(sodium_match.group(1).replace(',', ''))
protein_target = float(protein_match.group(1)) if protein_match else None

rows = [
    {'disease':'Chronic Kidney Disease', 'nutrient':'Sodium', 'min-value':None,
     'max-value':sodium_limit, 'unit':'mg/day', 'age-group':'Adult', 'gender':'All'},
    {'disease':'Chronic Kidney Disease', 'nutrient':'Protein', 'min-value':None,
     'max-value':None, 'unit':'g/kg/day', 'age-group':'Adult', 'gender':'All'},
    {'disease':'Chronic Kidney Disease', 'nutrient':'Potassium', 'min-value':None,
     'max-value':None, 'unit':'mg/day', 'age-group':'Adult', 'gender':'All'},
    {'disease':'Chronic Kidney Disease', 'nutrient':'Phosphorus', 'min-value':None,
     'max-value':None, 'unit':'mg/day', 'age-group':'Adult', 'gender':'All'},
]
df = pd.DataFrame(rows, columns=['disease','nutrient','min-value','max-value','unit','age-group','gender'])
display(df)

In [ ]:
evidence = [
    {'nutrient':'Sodium', 'recommendation_type':'upper_limit', 'value':sodium_limit,
     'unit':'mg/day', 'source_text':sodium_match.group(0), 'source_url':URL,
     'review_status':'Needs clinical review; patient-specific target may differ'},
    {'nutrient':'Protein', 'recommendation_type':'target_not_bound', 'value':protein_target,
     'unit':'g/kg/day', 'source_text':protein_match.group(0) if protein_match else '', 'source_url':URL,
     'review_status':'Needs clinical review; depends on patient and dialysis status'},
    {'nutrient':'Potassium', 'recommendation_type':'individualized_no_universal_number', 'value':None,
     'unit':'mg/day', 'source_text':'As CKD progresses, the next steps may include limiting dietary phosphorus and potassium.',
     'source_url':URL, 'review_status':'Do not infer a numeric bound'},
    {'nutrient':'Phosphorus', 'recommendation_type':'individualized_no_universal_number', 'value':None,
     'unit':'mg/day', 'source_text':'As CKD progresses, the next steps may include limiting dietary phosphorus and potassium.',
     'source_url':URL, 'review_status':'Do not infer a numeric bound'},
]
evidence_df = pd.DataFrame(evidence)
display(evidence_df)

In [ ]:
df.to_csv('niddk_ckd_nutrients.csv', index=False)
evidence_df.to_csv('niddk_ckd_evidence.csv', index=False)
print('Saved niddk_ckd_nutrients.csv and niddk_ckd_evidence.csv')